# FitDiT African Style Virtual Try-On

This notebook runs the FitDiT virtual try-on model and exposes it as a REST API


## Notes

- Enable GPU in Kaggle: Settings > Accelerator > GPU T4 x2
- Keep the notebook running to maintain the ngrok tunnel
- The model takes 2-3 minutes to load on first run after text encoders are cached
- Valid garment categories: `Upper-body`, `Lower-body`, `Dresses`

# **Installing all dependencies**

In [ ]:
import subprocess, sys

def pip(*args):
    # stderr is visible so installation errors are not hidden
    subprocess.check_call([sys.executable, '-m', 'pip'] + list(args))

# Step 1: Remove conflicting packages
print('Removing conflicting packages...')
subprocess.call(
    [sys.executable, '-m', 'pip', 'uninstall', '-y',
     'torch', 'torchvision', 'torchaudio', 'transformers',
     'diffusers', 'accelerate', 'huggingface_hub', 'peft', 'gradio'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Step 2: Install PyTorch first on its own before anything else
# This avoids the mid-session Python environment crash
print('Installing PyTorch 2.4.0 + CUDA 12.4...')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'torch==2.4.0', 'torchvision==0.19.0',
    '--index-url', 'https://download.pytorch.org/whl/cu124',
    '--quiet'
])
print('PyTorch installed.')

# Step 3: Install HuggingFace stack
print('Installing HuggingFace stack...')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'transformers==4.39.3',
    'diffusers==0.31.0',
    'accelerate==0.33.0',
    'huggingface_hub==0.26.5',
    'peft==0.9.0',
    '--quiet'
])
print('HuggingFace stack installed.')

# Step 4: Install supporting libraries
print('Installing supporting libraries...')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'gradio==4.44.1',
    'scikit-image==0.24.0',
    'onnxruntime==1.20.1',
    'opencv-python',
    'matplotlib==3.8.3',
    'einops==0.7.0',
    'numpy==1.26.4',
    'pymatting',
    'pyngrok',
    'fastapi',
    'uvicorn',
    'python-multipart',
    'requests',
    '--quiet'
])
print('Supporting libraries installed.')

# Step 5: Install rembg without deps to prevent numpy upgrade
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rembg', '--no-deps', '--quiet'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pooch', 'scipy', '--quiet'])
print('rembg installed.')


Removing conflicting packages...


# **Verifying package versions**

In [1]:
import torch, transformers, diffusers, accelerate, peft, numpy

print(f'PyTorch:      {torch.__version__}')
print(f'CUDA:         {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:          {torch.cuda.get_device_name(0)}')
print(f'Transformers: {transformers.__version__}')
print(f'Diffusers:    {diffusers.__version__}')
print(f'Accelerate:   {accelerate.__version__}')
print(f'PEFT:         {peft.__version__}')
print(f'NumPy:        {numpy.__version__}')

assert torch.__version__.startswith('2.4'), 'Wrong PyTorch version. Re-run Cell 1.'
assert torch.cuda.is_available(), 'CUDA not available. Enable GPU: Settings > Accelerator > GPU T4 x2'
assert numpy.__version__ == '1.26.4', 'Wrong numpy version. Re-run Cell 1.'
print('\nAll versions correct.')

PyTorch:      2.4.0+cu124
CUDA:         True
GPU:          Tesla T4
Transformers: 4.39.3
Diffusers:    0.31.0
Accelerate:   0.33.0
PEFT:         0.9.0
NumPy:        1.26.4

All versions correct.


# **Cloning the FitDiT repository**

In [2]:
import os

FITDIT_DIR = '/kaggle/working/FitDiT'

if not os.path.exists(FITDIT_DIR):
    print('Cloning FitDiT repository...')
    os.system(f'git clone https://github.com/BoyuanJiang/FitDiT.git {FITDIT_DIR}')
else:
    print('Repository already exists, skipping clone.')

os.chdir(FITDIT_DIR)
assert os.path.exists('gradio_sd3.py'), 'gradio_sd3.py not found. Re-clone the repo.'
print(f'Working directory: {os.getcwd()}')

Cloning FitDiT repository...


Cloning into '/kaggle/working/FitDiT'...


Working directory: /kaggle/working/FitDiT


# **Downloading FitDiT model weights**

In [3]:
from huggingface_hub import snapshot_download
import os

os.chdir('/kaggle/working/FitDiT')
MODEL_DIR = './FitDiT_model'

# Check if weights are already present
required_subdirs = ['transformer_garm', 'transformer_vton', 'vae']
already_downloaded = all(
    os.path.exists(os.path.join(MODEL_DIR, d)) and
    any(f.endswith(('.bin', '.safetensors')) for f in os.listdir(os.path.join(MODEL_DIR, d)))
    for d in required_subdirs
)

if already_downloaded:
    print('FitDiT weights already downloaded.')
else:
    print('Downloading FitDiT weights (~8GB). This takes 5-10 minutes...')
    snapshot_download(
        repo_id='BoyuanJiang/FitDiT',
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
    )
    print('Download complete.')

# Confirm structure
print('\nModel structure:')
for subdir in required_subdirs:
    files = [f for f in os.listdir(os.path.join(MODEL_DIR, subdir)) if f.endswith(('.bin', '.safetensors'))]
    print(f'  {subdir}: {files}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:834: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

dw-ll_ucoco_384.onnx:   0%|          | 0.00/134M [00:00<?, ?B/s]

...

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/739 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Download complete.

Model structure:
  transformer_garm: ['diffusion_pytorch_model.safetensors']
  transformer_vton: ['diffusion_pytorch_model.safetensors']
  vae: ['diffusion_pytorch_model.safetensors']


# **Downloading SD3 text encoders**

In [ ]:
from huggingface_hub import snapshot_download, login
import os

HF_TOKEN = os.getenv('HF_TOKEN') or os.getenv('HUGGING_FACE_HUB_TOKEN')
if not HF_TOKEN:
    raise ValueError('Set HF_TOKEN (or HUGGING_FACE_HUB_TOKEN) in your environment before running this cell.')

login(token=HF_TOKEN, add_to_git_credential=False)
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

print('Downloading SD3 text encoders (~9GB). This takes 5-10 minutes...')
print('This only needs to run once. After this, model loading takes under 1 minute.')

snapshot_download(
    repo_id='stabilityai/stable-diffusion-3-medium-diffusers',
    token=HF_TOKEN,
    local_dir_use_symlinks=False,
    allow_patterns=[
        'text_encoder/**',
        'text_encoder_2/**',
        'text_encoder_3/**',
        'tokenizer/**',
        'tokenizer_2/**',
        'tokenizer_3/**',
    ],
)


This only needs to run once. After this, model loading takes under 1 minute.


Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.39G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/247M [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

'/root/.cache/huggingface/hub/models--stabilityai--stable-diffusion-3-medium-diffusers/snapshots/ea42f8cef0f178587cf766dc8129abd379c90671'

# **Verifying imports and applying patches**

In [5]:
import sys, os
sys.path.insert(0, '/kaggle/working/FitDiT/src')
os.chdir('/kaggle/working/FitDiT')

# Verify the core pipeline class imports correctly
try:
    from pipeline_stable_diffusion_3_tryon import StableDiffusion3TryOnPipeline
    print('StableDiffusion3TryOnPipeline imported successfully.')
except Exception as e:
    print(f'Pipeline import failed: {e}')
    raise

# Patch gradio_client to handle boolean JSON schemas
# FitDiT passes True/False as schema values, but gradio_client
utils_path = '/usr/local/lib/python3.12/dist-packages/gradio_client/utils.py'
with open(utils_path, 'r') as f:
    content = f.read()

target = 'def _json_schema_to_python_type(schema: Any, defs: dict[str, Any] | None = None) -> str:'
replacement = '''def _json_schema_to_python_type(schema: Any, defs: dict[str, Any] | None = None) -> str:
    if not isinstance(schema, dict):
        return "any"'''

add_props_old = "f\"str, {_json_schema_to_python_type(schema['additionalProperties'], defs)}\""
add_props_new = "f\"str, {_json_schema_to_python_type(schema['additionalProperties'] if isinstance(schema.get('additionalProperties'), dict) else {}, defs)}\""

patched = 0
if target in content and 'if not isinstance(schema, dict)' not in content:
    content = content.replace(target, replacement, 1)
    patched += 1
if add_props_old in content:
    content = content.replace(add_props_old, add_props_new, 1)
    patched += 1

with open(utils_path, 'w') as f:
    f.write(content)

if patched > 0:
    print(f'gradio_client patched ({patched} fixes applied).')
else:
    print('gradio_client already patched.')



2026-03-08 19:37:02.028980: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772998622.261273     162 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772998622.389562     162 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772998623.107409     162 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772998623.107444     162 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772998623.107447     162 computation_placer.cc:177] computation placer alr

StableDiffusion3TryOnPipeline imported successfully.
gradio_client patched (1 fixes applied).


In [6]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pymatting', '-q'])

0

# **Load preprocessing models**

In [7]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import tempfile, os
from PIL import Image
from rembg import remove
from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation

print('Loading SegFormer clothing segmentation model...')
seg_processor = AutoImageProcessor.from_pretrained('mattmdjaga/segformer_b2_clothes')
seg_model = AutoModelForSemanticSegmentation.from_pretrained('mattmdjaga/segformer_b2_clothes')
print('SegFormer loaded.')


def _run_segformer(img: Image.Image):
    """
    Internal helper. Runs SegFormer on an image and returns
    the per-pixel class predictions as a numpy array.

    SegFormer class indices:
      0=background, 1=hat, 2=hair, 3=sunglasses, 4=upper-clothes,
      5=skirt, 6=pants, 7=dress, 8=belt, 9=left-shoe, 10=right-shoe,
      11=face, 12=left-leg, 13=right-leg, 14=left-arm, 15=right-arm,
      16=bag, 17=scarf
    """
    inputs = seg_processor(images=img, return_tensors='pt')
    with torch.no_grad():
        outputs = seg_model(**inputs)

    logits_resized = F.interpolate(
        outputs.logits,
        size=(img.height, img.width),
        mode='bilinear',
        align_corners=False
    )
    predictions = logits_resized.argmax(1).squeeze().cpu().numpy()
    probs = F.softmax(logits_resized, dim=1).squeeze().cpu().numpy()
    return predictions, probs


def analyze_garment_skin_exposure(garment_img: Image.Image) -> dict:
    """
    Analyzes the garment photo to detect what body parts will be
    visible when wearing the garment. This is used to prepare the
    person photo so it matches what the garment naturally exposes.

    Returns a dict with:
      shows_legs       -- True if the garment shows bare legs
      shows_arms       -- True if the garment shows bare arms
      has_skirt_or_dress -- True if garment is a skirt or dress
      leg_coverage     -- float 0-1, how much of the legs are visible
    """
    predictions, _ = _run_segformer(garment_img)

    has_visible_legs = (
        (predictions == 12).sum() > 300 or
        (predictions == 13).sum() > 300
    )
    has_visible_arms = (
        (predictions == 14).sum() > 300 or
        (predictions == 15).sum() > 300
    )
    has_skirt_or_dress = (
        (predictions == 5).sum() > 500 or
        (predictions == 7).sum() > 500
    )

    # Estimate how much of the legs are exposed
    leg_pixels = np.where((predictions == 12) | (predictions == 13))
    leg_coverage = 0.0
    if len(leg_pixels[0]) > 0:
        leg_top = leg_pixels[0].min()
        leg_bottom = leg_pixels[0].max()
        leg_coverage = (leg_bottom - leg_top) / garment_img.height

    analysis = {
        'shows_legs': has_visible_legs,
        'shows_arms': has_visible_arms,
        'has_skirt_or_dress': has_skirt_or_dress,
        'leg_coverage': round(leg_coverage, 2),
    }
    print(f'  Garment analysis: {analysis}')
    return analysis


def prepare_person_for_garment(
    person_img: Image.Image,
    garment_analysis: dict,
    category: str
) -> Image.Image:
    """
    Adjusts the person photo to match what the garment exposes.

    If the garment is a dress/skirt that shows legs but the person
    is wearing trousers, replaces the trouser region with the
    person's own skin tone so FitDiT renders the legs correctly.

    Skin tone is sampled from the person's visible face/arm pixels
    so the result matches their actual complexion.
    """
    # Upper-body garments never conflict with what the person wears below
    if category == 'Upper-body':
        return person_img

    # If the garment does not show legs, no adjustment needed
    if not garment_analysis['shows_legs'] and not garment_analysis['has_skirt_or_dress']:
        print('  No leg conflict — person photo unchanged.')
        return person_img

    original_array = np.array(person_img)
    predictions, _ = _run_segformer(person_img)

    # Check if the person is wearing trousers
    trouser_mask = (predictions == 6)
    if trouser_mask.sum() < 500:
        print('  Person legs already suitable for this garment.')
        return person_img

    print(f'  Garment shows legs but person has trousers ({trouser_mask.sum()} px).')
    print('  Replacing trousers with person skin tone...')

    # Sample the person's skin tone from their visible skin pixels
    # Priority: legs > arms > face (legs give the most accurate leg skin tone)
    skin_classes = [12, 13, 14, 15, 11]  # legs, arms, face
    skin_mask = np.zeros(predictions.shape, dtype=bool)
    for class_idx in skin_classes:
        if class_idx <= predictions.max():
            skin_mask |= (predictions == class_idx)

    if skin_mask.sum() > 100:
        skin_pixels = original_array[skin_mask]
        skin_color = skin_pixels.mean(axis=0).astype(np.uint8)
        print(f'  Skin tone sampled from person: RGB{tuple(skin_color)}')
    else:
        # Fallback if no skin is visible in the photo
        skin_color = np.array([180, 140, 110], dtype=np.uint8)
        print('  No skin detected, using fallback skin tone.')

    # Fill trouser region with skin color plus subtle noise for realism
    result_array = original_array.copy()
    trouser_coords = np.where(trouser_mask)
    num_pixels = len(trouser_coords[0])
    noise = np.random.randint(-12, 12, (num_pixels, 3))
    skin_fill = np.clip(skin_color.astype(int) + noise, 0, 255).astype(np.uint8)
    result_array[trouser_mask] = skin_fill

    # Smooth the boundary between the skin fill and the original image
    # to avoid a hard visible line at the trouser edges
    mask_uint8 = trouser_mask.astype(np.uint8) * 255
    blurred_mask = cv2.GaussianBlur(mask_uint8.astype(float), (21, 21), 0) / 255.0
    for c in range(3):
        result_array[:, :, c] = (
            blurred_mask * result_array[:, :, c] +
            (1 - blurred_mask) * original_array[:, :, c]
        ).astype(np.uint8)

    print('  Trouser replacement done.')
    return Image.fromarray(result_array)


def remove_person_background(person_img: Image.Image) -> Image.Image:
    """
    Removes the background from a person photo and replaces it
    with white. Uses the rembg library (U2Net model).
    FitDiT produces better results with a clean white background.
    """
    rgba = remove(person_img)
    white_bg = Image.new('RGB', rgba.size, (255, 255, 255))
    white_bg.paste(rgba, mask=rgba.split()[3])
    return white_bg


def clean_garment_image(garment_img: Image.Image) -> Image.Image:
    """
    Extracts just the garment fabric from a photo, removing skin
    and background. Uses SegFormer for garment detection combined
    with color-based skin detection in HSV and YCrCb color spaces.

    This prevents visible skin or background from the garment photo
    bleeding into the try-on result.
    """
    original_array = np.array(garment_img)
    predictions, probs = _run_segformer(garment_img)

    # Build garment mask: upper-clothes, skirt, pants, dress, belt
    garment_classes = [4, 5, 6, 7, 8]
    garment_mask = np.zeros_like(predictions, dtype=bool)
    for class_idx in garment_classes:
        if class_idx <= predictions.max():
            class_mask = predictions == class_idx
            if class_mask.sum() > 100:
                garment_mask |= class_mask

    # Build skin mask from SegFormer predictions
    skin_classes = [11, 12, 13, 14, 15]
    skin_mask = np.zeros_like(predictions, dtype=bool)
    for class_idx in skin_classes:
        if class_idx <= predictions.max():
            skin_mask |= (predictions == class_idx)

    # Supplement with color-based skin detection to catch missed skin tones
    hsv = cv2.cvtColor(original_array, cv2.COLOR_RGB2HSV_FULL)
    ycrcb = cv2.cvtColor(original_array, cv2.COLOR_RGB2YCrCb)
    hsv_skin = cv2.inRange(hsv,
        np.array([0, 25, 40], dtype=np.uint8),
        np.array([30, 180, 220], dtype=np.uint8))
    ycrcb_skin = cv2.inRange(ycrcb,
        np.array([0, 135, 85], dtype=np.uint8),
        np.array([255, 180, 135], dtype=np.uint8))
    color_skin = (cv2.bitwise_and(hsv_skin, ycrcb_skin) > 0)

    # Only remove color-detected skin where garment confidence is low
    # This avoids removing warm-toned African print fabric
    clothing_prob = np.zeros_like(predictions, dtype=float)
    for class_idx in [4, 5, 6, 7]:
        if class_idx < probs.shape[0]:
            clothing_prob = np.maximum(clothing_prob, probs[class_idx])
    combined_skin = skin_mask | (color_skin & (clothing_prob < 0.2))

    # Final mask: garment pixels minus skin pixels
    final_mask = garment_mask & (~combined_skin)

    # Clean up mask edges with morphological operations
    mask_uint8 = final_mask.astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_uint8 = cv2.morphologyEx(mask_uint8, cv2.MORPH_CLOSE, kernel)
    mask_uint8 = cv2.morphologyEx(mask_uint8, cv2.MORPH_OPEN, kernel)
    final_mask = mask_uint8 > 0

    white_bg = np.ones_like(original_array) * 255
    white_bg[final_mask] = original_array[final_mask]
    return Image.fromarray(white_bg.astype(np.uint8))


def preprocess_images(
    person_img: Image.Image,
    garment_img: Image.Image,
    category: str = 'Upper-body'
):
    """
    Full preprocessing pipeline. Runs all steps in order and saves
    results to temp files on disk.

    FitDiT's process() method requires file paths, not PIL Images,
    so we save the preprocessed images before passing them in.

    Returns (person_path, garment_path, clean_person, clean_garment)
    where the paths are temp file paths and the images are PIL Images
    for display purposes.

    Steps:
      1. Analyze garment to detect what body parts it exposes
      2. Adjust person photo to match (replace trousers if needed)
      3. Remove person background
      4. Clean garment image
      5. Save both to temp files
    """
    print('  Analyzing garment...')
    garment_analysis = analyze_garment_skin_exposure(garment_img)

    print('  Preparing person photo...')
    person_img = prepare_person_for_garment(person_img, garment_analysis, category)

    print('  Removing person background...')
    clean_person = remove_person_background(person_img)

    print('  Cleaning garment image...')
    clean_garment = clean_garment_image(garment_img)

    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as pf:
        clean_person.save(pf.name, format='JPEG', quality=95)
        person_path = pf.name

    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as gf:
        clean_garment.save(gf.name, format='JPEG', quality=95)
        garment_path = gf.name

    return person_path, garment_path, clean_person, clean_garment




Loading SegFormer clothing segmentation model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration. Please open a PR/issue to update `preprocessor_config.json` to use `image_processor_type` instead of `feature_extractor_type`. This warning will be removed in v4.40.
/usr/local/lib/python3.12/dist-packages/transformers/models/segformer/image_processing_segformer.py:103: FutureWarning: The `reduce_labels` parameter is deprecated and will be removed in a future version. Please use `do_reduce_labels` instead.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/109M [00:00<?, ?B/s]

SegFormer loaded.


# **Starting the FastAPI server**

In [8]:
import sys, os, subprocess, time, threading, io, tempfile
sys.path.insert(0, '/kaggle/working/FitDiT/src')
os.chdir('/kaggle/working/FitDiT')

import torch
import numpy as np
from fastapi import FastAPI, File, UploadFile, HTTPException, Form
from fastapi.responses import StreamingResponse
import uvicorn
from pyngrok import ngrok
from PIL import Image

# Kill any previous server running on this port
subprocess.call(['fuser', '-k', '8000/tcp'], stderr=subprocess.DEVNULL)
time.sleep(2)

app = FastAPI(title='FitDiT African Style Try-On API')
generator = None
load_error = None


def load_pipeline():
    """
    Load FitDiTGenerator in a background thread so the server
    can start accepting health check requests immediately.
    """
    global generator, load_error
    try:
        print('Loading FitDiTGenerator...')
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            'gradio_sd3', '/kaggle/working/FitDiT/gradio_sd3.py')
        mod = importlib.util.module_from_spec(spec)
        mod.__file__ = '/kaggle/working/FitDiT/gradio_sd3.py'
        spec.loader.exec_module(mod)

        # offload=True moves unused components to CPU to save GPU memory
        # with_fp16=True uses half precision for faster inference on T4
        generator = mod.FitDiTGenerator(
            './FitDiT_model',
            offload=True,
            with_fp16=True,
            device='cuda:0',
        )
        print('Generator ready.')
    except Exception as e:
        import traceback
        load_error = traceback.format_exc()
        print(f'Load failed:\n{load_error}')


threading.Thread(target=load_pipeline, daemon=True).start()


@app.get('/health')
async def health():
    return {'model_ready': generator is not None, 'error': load_error}


@app.get('/')
async def root():
    return {'status': 'running', 'model_ready': generator is not None}


@app.post('/tryon')
async def tryon(
    person: UploadFile = File(...),
    garment: UploadFile = File(...),
    category: str = Form(default='Upper-body'),
    n_steps: int = Form(default=20),
    image_scale: float = Form(default=2.5),
    seed: int = Form(default=42),
    resolution: str = Form(default='768x1024'),
):
    if generator is None:
        detail = f'Load error: {load_error}' if load_error else 'Model still loading. Check /health.'
        raise HTTPException(status_code=503, detail=detail)

    try:
        person_img = Image.open(io.BytesIO(await person.read())).convert('RGB')
        garment_img = Image.open(io.BytesIO(await garment.read())).convert('RGB')

        # Preprocess both images before passing to FitDiT.
        # preprocess_images() is defined in Cell 7 and:
        #   - analyzes garment to detect what body parts it exposes
        #   - replaces person trousers with skin if garment shows legs
        #   - removes person background
        #   - cleans garment image
        print('Preprocessing images...')
        person_path, garment_path, _, _ = preprocess_images(person_img, garment_img, category)

        person_np = np.array(Image.open(person_path).convert('RGB'))

        # Step 1: Generate pose keypoints from the person image.
        # dwprocessor expects a numpy array and returns a tuple — use index 0.
        print('  Generating pose...')
        pose_image = generator.dwprocessor(person_np)[0]

        # Step 2: Generate the clothing mask for the target body region.
        # generate_mask expects a file path and returns a tuple — use index 0.
        # Index 0 is a dict with keys: background, layers, composite.
        print('  Generating mask...')
        pre_mask = generator.generate_mask(
            person_path,
            category=category,
            offset_top=0, offset_bottom=0,
            offset_left=0, offset_right=0
        )[0]

        # Step 3: Run the try-on diffusion.
        # process() expects file paths for vton_img and garm_img, not PIL Images.
        print('  Running try-on inference...')
        with torch.inference_mode():
            result = generator.process(
                vton_img=person_path,
                garm_img=garment_path,
                pre_mask=pre_mask,
                pose_image=pose_image,
                n_steps=n_steps,
                image_scale=image_scale,
                seed=seed,
                num_images_per_prompt=1,
                resolution=resolution,
            )

        os.unlink(person_path)
        os.unlink(garment_path)

        out_img = result[0] if isinstance(result, (list, tuple)) else result
        buf = io.BytesIO()
        out_img.save(buf, format='JPEG', quality=95)
        buf.seek(0)
        print('  Done.')
        return StreamingResponse(buf, media_type='image/jpeg')

    except Exception as e:
        import traceback
        raise HTTPException(status_code=500, detail=traceback.format_exc())



NGROK_TOKEN = '39s5Q7JVNNTMGYxf7W9wkKe5eHU_7Ltt4Q2C8My5TY7sE6z3J'
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
time.sleep(1)

threading.Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'),
    daemon=True
).start()
time.sleep(3)

api_url = ngrok.connect(8000, 'http')
BASE_URL = str(api_url).split('"')[1] if '"' in str(api_url) else str(api_url)

print(f'\nAPI URL:   {BASE_URL}')
print(f'Health:    {BASE_URL}/health')
print(f'Try-on:    {BASE_URL}/tryon')
print(f'\nFlutter base URL:')
print(f'  const apiUrl = "{BASE_URL}";')
print('\nPolling health every 20 seconds...')

import requests as req
for i in range(25):
    time.sleep(20)
    try:
        r = req.get('http://localhost:8000/health', timeout=5)
        data = r.json()
        print(f'  [{(i+1)*20}s] {data}')
        if data.get('model_ready'):
            print(f'\nModel ready. Send requests to: {BASE_URL}/tryon')
            break
        if data.get('error'):
            print(f'\nLoad error:\n{data["error"]}')
            break
    except Exception as e:
        print(f'  [{(i+1)*20}s] Waiting for server... ({e})')

Loading FitDiTGenerator...
                                                                                                    
API URL:   https://unexcepted-coaly-candie.ngrok-free.dev
Health:    https://unexcepted-coaly-candie.ngrok-free.dev/health
Try-on:    https://unexcepted-coaly-candie.ngrok-free.dev/tryon

Flutter base URL:
  const apiUrl = "https://unexcepted-coaly-candie.ngrok-free.dev";

Polling health every 20 seconds...
  [20s] {'model_ready': False, 'error': None}


/kaggle/working/FitDiT/gradio_sd3.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pose_guider.load_state_dict(torch.load(os.path.join(model_root, "pose_guider", "diffus

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

  [40s] {'model_ready': False, 'error': None}


...

  Person legs already suitable for this garment.
  Removing person background...


100%|████████████████████████████████████████| 176M/176M [00:00<00:00, 234GB/s]


  Cleaning garment image...
  Generating pose...
  Generating mask...
  Running try-on inference...


  0%|          | 0/20 [00:00<?, ?it/s]

  Done.
Preprocessing images...
  Analyzing garment...


# **Testing with real images**

In [ ]:
import os, shutil
import numpy as np, torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display


PERSON_INPUT  = '/kaggle/input/datasets/aminalawal/try-on/person.jpg'
GARMENT_INPUT = '/kaggle/input/datasets/aminalawal/try-on/garment.jpg'
CATEGORY      = 'Dresses'   # Upper-body | Lower-body | Dresses


# Copy uploaded files to working directory
PERSON_PATH  = '/kaggle/working/person.jpg'
GARMENT_PATH = '/kaggle/working/garment.jpg'
shutil.copy(PERSON_INPUT, PERSON_PATH)
shutil.copy(GARMENT_INPUT, GARMENT_PATH)
print('Images copied.')

person_img  = Image.open(PERSON_PATH).convert('RGB')
garment_img = Image.open(GARMENT_PATH).convert('RGB')

# Run full preprocessing pipeline — category is required so the
# garment analysis and trouser replacement work correctly
print('Preprocessing...')
person_path, garment_path, clean_person, clean_garment = preprocess_images(
    person_img, garment_img, CATEGORY
)

person_np = np.array(Image.open(person_path).convert('RGB'))

# Generate pose keypoints
print('Generating pose...')
pose_image = generator.dwprocessor(person_np)[0]

# Generate clothing mask
print('Generating mask...')
pre_mask = generator.generate_mask(
    person_path,
    category=CATEGORY,
    offset_top=0, offset_bottom=0, offset_left=0, offset_right=0
)[0]

# Run try-on inference
print('Running try-on (30-60 seconds)...')
with torch.inference_mode():
    result = generator.process(
        vton_img=person_path,
        garm_img=garment_path,
        pre_mask=pre_mask,
        pose_image=pose_image,
        n_steps=20,
        image_scale=2.5,
        seed=42,
        num_images_per_prompt=1,
        resolution='768x1024',
    )

# Cleanup temp files and save result
os.unlink(person_path)
os.unlink(garment_path)

out_img = result[0] if isinstance(result, (list, tuple)) else result
out_img.save('/kaggle/working/result.jpg')
print('Result saved to /kaggle/working/result.jpg')

# Display all four images side by side for comparison
fig, axes = plt.subplots(1, 4, figsize=(20, 8))
axes[0].imshow(person_img);   axes[0].set_title('Person (original)');  axes[0].axis('off')
axes[1].imshow(clean_person); axes[1].set_title('Person (processed)'); axes[1].axis('off')
axes[2].imshow(garment_img);  axes[2].set_title('Garment (original)'); axes[2].axis('off')
axes[3].imshow(out_img);      axes[3].set_title('Try-on result');       axes[3].axis('off')
plt.tight_layout()
plt.show()